# import #

In [89]:
import re
import sys
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

# shell call #

In [90]:
partner = 'efrapo'
base_dir = 'data'
data_dir = 'template'

if re.match('create_project_partner_crm_db.py', sys.argv[0]):
    if len(sys.argv) >= 1:
        partner = sys.argv[1]
    if len(sys.argv) >= 2:
        base_dir = sys.argv[2]
    if len(sys.argv) >= 3:
        data_dir = sys.argv[3]

folder = Path(base_dir) / data_dir
folder.mkdir(exist_ok=True)

filename = f"{folder}/{partner}_crm_db.xlsx" 

# load data #

## load project partner data ##

In [91]:
ppartner_df = pd.read_pickle('./data/projectpartners.pkl')
ppartner_df.head()

,PROJECT_PARTNER_NAME,MEMBER_ID,CUSTOMER_INTERNAL_ID,CUSTOMER_NAME
0,sefi,000100_FMEM,064,AMCOR
1,efrapo,000105_FMEM,064,AMCOR
2,polarbearings,009564_FMEM,207,TRIVIUM


In [92]:
customer_list = ppartner_df.loc[ppartner_df["PROJECT_PARTNER_NAME"] == partner, "CUSTOMER_INTERNAL_ID"].tolist()

print(customer_list)

if len(customer_list) == 0:
    sys.exit(0)

['064']


In [93]:
member_ids = ppartner_df.loc[ppartner_df["PROJECT_PARTNER_NAME"] == partner, "MEMBER_ID"]

member_id = member_ids.iloc[0] if not member_ids.empty else None

print(member_id)

if member_id == None:
    sys.exit(0)

000105_FMEM


## load master data ##

In [94]:
customers_df = pd.read_pickle('./data/customers.pkl')
customers_df.head()

,CUSTOMER_ID,CUSTOMER_NAME,CUSTOMER_INTERNAL_ID,CUSTOMER_FOLDER_NAME,PILOT,IS_ACTIVE,HAS_CONTRACT,HAS_RFQ
0,042316_FCLI,ABBOTT,217,Abbott 217,Jasper Groenendijk,yes,yes,yes
1,028099_FCLI,ADIENT,163,,Sylvie PIEROTTI,None,None,None
2,025681_FCLI,ADR GROUP,118,,Sylvie PIEROTTI,None,None,None
3,040007_FCLI,AGROFERT,263,Agrofert 263,Christophe PATRAULT,None,None,None
4,030720_FCLI,AKZO NOBEL,241,,Sylvie PIEROTTI,None,None,None


In [95]:
plants_df = pd.read_pickle('./data/plants.pkl')
plants_df.head()

,CUSTOMER_ID,PLANT_ID,PLANT_NAME,COUNTRY,CITY,SECTOR,CUSTOMER_INTERNAL_ID,PLANT_CLOSED
0,042316_FCLI,042319_FCLI,ABBOTT - Abbott Biologicals - NL - 8121 AA - OLST,NL,OLST,PHARMACEUTICAL,217,None
1,042316_FCLI,042318_FCLI,ABBOTT - Abbott Healthcare Products - NL - 138...,NL,WEESP,PHARMACEUTICAL,217,None
2,042316_FCLI,042317_FCLI,ABBOTT - Abbott Laboratories - DE - 31535 - NE...,DE,NEUSTADT AM RÜBENBERGE,PHARMACEUTICAL,217,None
3,042316_FCLI,042320_FCLI,ABBOTT - Abbott Laboratories - NL - 8041 AK - ...,NL,ZWOLLE,PHARMACEUTICAL,217,None
4,005343_FCLI,030987_FCLI,AMCOR - Amcor Flexibles - BE - 6031 - Monceau-...,BE,MONCEAU-SUR-SAMBRE,PACKAGING (PLASTIC AND CARDBOARD),064,YES


In [96]:
productfamilies_df = pd.read_pickle('./data/productfamilies.pkl')
productfamilies_df.head()

,PRODUCTFAMILY_ID,PRODUCTFAMILY_CODE,PRODUCTFAMILY_NAME
0,01_Roulements,01,"Bearings (bearing, housing)"
1,02_Transmission mecanique,02,Mechanical Transmission
2,03_Transmission electromecanique,03,Electromechanically transmission (motors …)
3,04_Guidage,04,Linear motion
4,05_Etancheite,05,Sealing


In [97]:
branches_df = pd.read_pickle('./data/branches.pkl')
branches_df.head()

,MEMBER_ID,BRANCH_ID,BRANCH_NAME,MEMBER_NAME,BRANCH_CLOSED
0,000099_FMEM,CIRALB,CIR Albi,CIR,None
1,000099_FMEM,CIRBEZ,CIR Beziers,CIR,None
2,000099_FMEM,CIRMER,CIR Bordeaux,CIR,None
3,000099_FMEM,CIRCAR,CIR Carcassonne,CIR,None
4,000099_FMEM,CIRCAS,CIR Castres,CIR,None


In [98]:
suppliers_df = pd.read_pickle('./data/suppliers.pkl')
suppliers_df.head()

,SUPPLIER_ID,SUPPLIER_NAME
0,000436_FFOUR,DIVERS/OTHER
1,000094_FFOUR,3 M
2,000095_FFOUR,ABA FRANCE (SERFLEX)= NORMA
3,031147_FFOUR,ABB
4,000096_FFOUR,ACC


# get reduced master data #

In [99]:
customers_red_df = customers_df[customers_df['CUSTOMER_INTERNAL_ID'].isin(customer_list)]
customers_red_df.head()

,CUSTOMER_ID,CUSTOMER_NAME,CUSTOMER_INTERNAL_ID,CUSTOMER_FOLDER_NAME,PILOT,IS_ACTIVE,HAS_CONTRACT,HAS_RFQ
8,005343_FCLI,AMCOR,064,Amcor 064,Sylvie PIEROTTI,yes,yes,yes


In [100]:
plants_red_df = plants_df[plants_df['CUSTOMER_INTERNAL_ID'].isin(customer_list)]
plants_red_df.head()

,CUSTOMER_ID,PLANT_ID,PLANT_NAME,COUNTRY,CITY,SECTOR,CUSTOMER_INTERNAL_ID,PLANT_CLOSED
4,005343_FCLI,030987_FCLI,AMCOR - Amcor Flexibles - BE - 6031 - Monceau-...,BE,MONCEAU-SUR-SAMBRE,PACKAGING (PLASTIC AND CARDBOARD),064,YES
5,005343_FCLI,031011_FCLI,AMCOR - Amcor Flexibles - CH - 3401 - Burgdorf,CH,BURGDORF,PACKAGING (PLASTIC AND CARDBOARD),064,None
6,005343_FCLI,031009_FCLI,AMCOR - Amcor Flexibles - CH - 8280 - Kreuzlingen,CH,KREUZLINGEN,PACKAGING (PLASTIC AND CARDBOARD),064,None
7,005343_FCLI,031010_FCLI,AMCOR - Amcor Flexibles - CH - 9403 - Goldach ...,CH,GOLDACH,PACKAGING (PLASTIC AND CARDBOARD),064,None
8,005343_FCLI,030988_FCLI,AMCOR - Amcor Flexibles - DE - 78224 - Singen,DE,SINGEN,PACKAGING (PLASTIC AND CARDBOARD),064,None


In [101]:
branches_red_df = branches_df[branches_df['MEMBER_ID'] == member_id]
branches_red_df.head()

,MEMBER_ID,BRANCH_ID,BRANCH_NAME,MEMBER_NAME,BRANCH_CLOSED
17,000105_FMEM,EFRAPO25,Efrapo - Besançon,EFRAPO,None
18,000105_FMEM,EFRAPO21,Efrapo - Dijon,EFRAPO,None
19,000105_FMEM,EFRAPO88,Efrapo - Epinal,EFRAPO,None
20,000105_FMEM,EFRAPO57,Efrapo - Metz,EFRAPO,None
21,000105_FMEM,EFRAPO68,Efrapo - Mulhouse,EFRAPO,None


# create project partner crm_db #

In [102]:
with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    customers_red_df.to_excel(writer, sheet_name="MD_CUSTOMER", index=False)
    plants_red_df.to_excel(writer, sheet_name="MD_PLANTS", index=False)
    branches_red_df.to_excel(writer, sheet_name="MD_BRANCHES", index=False)
    suppliers_df.to_excel(writer, sheet_name="MD_SUPPLIERS", index=False)    
    productfamilies_df.to_excel(writer, sheet_name="MD_PRODUCTFAMILIES", index=False)

In [103]:
wb = load_workbook(filename)

for ws in wb.worksheets: 
    for col in ws.columns:
        max_length = 0
        col_letter = col[0].column_letter

        for cell in col:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[col_letter].width = max_length + 4

wb.save(filename)

print(f"created file {filename}")